# Notebook of comparison between Abromics Kg powered by SOSA or RDF data cube as core ontologies

The goal of this notebook is to compare the capacity for the SOSA and RDF data cube ontologies to store data from an ABRomics report.

The relevant points for this comparison will be: 

- The capacity to store target antibiotics and resistance gene of a single ABRomics report while respecting the classes defined by the ontology
- The query complexity when doing a simple request
- The query performance
- The hability to link the data from the report to other specific ontologies

These 4 points will be tested for the **SOSA/SSN** ontology and the **RDF data cube** ontology

### Summary

- 1. [Necessary imports](#necessaryImports) 
- 2. [Utility classes](#utilityClasses)
    - 1. [Benchmark](#benchmark)
- 3. [RDF data cube analysis](#rdfDatacubeAnalysis)
    - 1. [QB General informations](#QB-generalInfos)
    - 2. [QB Storing ABRomics report data](#QB-storingReportData)
    - 3. [QB Query complexity](#QB-queryComplexity)
    - 4. [QB Query performance](#QB-queryPerformance)
    - 5. [QB Linking to other ontologies](#QB-linkingToOtherOntologies)
    - 6. [Observations](#QB-observations)
- 4. [SOSA/SSN analysis](#sosaAnalysis)
    - 1. [SOSA General informations](#SOSA-generalInfos)
    - 2. [SOSA Storing ABRomics report data](#SOSA-storingReportData)
    - 3. [SOSA Query complexity](#SOSA-queryComplexity)
    - 4. [SOSA Query performance](#SOSA-queryPerformance)
    - 5. [SOSA Linking to other ontologies](#SOSA-linkingToOtherOntologies)
    - 6. [Observations](#SOSA-observations)
- 5. [Conclusion](#conclusion)

## Necessary imports <a name="introduction"></a>

In [ ]:
# imports

import json
import time
import pandas as pd

from rdflib import Graph
import uuid
from rdflib import Namespace
from rdflib import FOAF, DCTERMS, SKOS, VANN, TIME, RDF, RDFS, XSD, OWL, Literal

In [ ]:
# Load the local ABRomics report
with open("reports.json", "r") as f:
    reports = json.load(f)

## Utility classes

### Benchmark utility class <a name="benchmark"></a> 

The benchmark class is a utility that allows to perform query benchmarks in bulk for a specific graph and a set of queries.
The result of the benchmark can be compared using this utility class as the process is the same whatever the graph and the queries are.

#### How to use it ?

- 1. Instanciate the Benchmarck class with the following parameters:

    - **graph**     : RDFlib graph object in which the benchmark should be performed
    - **queries**   : A list of queries and queries description that should be run in bulk in the benchmark

```python
queries = [
    {
        "description": "description of the query",
        "content": "content of the query itself"
    }
]
```

- 2. Call the *launch()* method to lanch the benchmark. All the queries indicated in the **queries** parameter are launched 100 times each and preformance are measured
- 3. Call the *displayResult()* method to get a table with the measured performances for all the querues

In [ ]:
## Benchmark class utility class that allows to have all the tools to benchmark graphs and queries

class Benchmark:
    
    def __init__(self, graph, queries):
        self.graph = graph
        self.queries = queries
        self.benchmarkData = [[], [], [], []]
        self.benchmarkRowNames = ["description", "avg_elapsed_time (ms)", "max_elapsed_time (ms)", "min_elapse_time (ms)"]
        self.benchmarkDf = pd.DataFrame()
    
    def launch(self):
        for query in queries:
            times = []
            for i in range(0, 100):
                startTime = time.time()
                res = self.graph.query(query["content"])
                endTime = time.time() - startTime
                times.append(round(endTime * 1000, 2))
            self.benchmarkData[0].append(query["description"])
            self.benchmarkData[1].append(round(sum(times) / len(times), 2))
            self.benchmarkData[2].append(max(times))
            self.benchmarkData[3].append(min(times))
        self.benchmarkDf = pd.DataFrame(self.benchmarkData, index=self.benchmarkRowNames, columns=[f"query {i}" for i in range(1, len(self.queries) + 1)])
        
    def displayResults(self):
        print(self.benchmarkDf)
        


## RDF data cube anlalysis <a name="rdfDataCubeAnalysis"></a>

### QB General informations <a name="QB-generalInfos"></a>

The QB ontology is used to store large dataset that could be represented as a matrix of n dimensions. Oriented towards statistical usecases, RDF data cube define different classes to interact with the data cube using slice. Filters can also be applied to the properties that are linked to the slice or to the dataset structure definition

<table><tr>
<td> <img src="img/qb-schema.png" style="width: 900px; height:800px"/> </td>
<td> <img src="img/data-cube-illustration.jpeg" style="width: 900px; height:800px"/> </td>
</tr></table>

### QB Storing ABRomics data report <a name="QB-storingReportData"></a>

The ABRomics report can be stored in a RDF data cube data set where the content of the data is stored in many observations.
As the ABRomics report is a simple table, defining slices is useless because slices are used to fix two dimensions of the data cube to extract a table that can later be analysed.

The code below perform an instanciation of QB schema by storing the resistance genes and target antibiotic found in one ABRomics report. The same storing strategy can be applied to any data present in the ABRomics report even with a datacube composed of multiple reports.  

In [ ]:
g = Graph()

ABROMICS = Namespace("http://www.abromics.org/ontology/")
RDFS = Namespace("http://www.w3.org/2000/01/rdf-schema#")
DCT = Namespace("http://purl.org/dc/terms/")

QB = Namespace("http://purl.org/linked-data/cube#")

# Read part 6 of the RDF data cube documentation to understand how such data should be encoded

analysisResult = pd.DataFrame(reports[0]["sections"][2]["data"][0]["values"], index=reports[0]["sections"][2]["data"][0]["header"])

# Create all the dimensions and measures
resistanceGenePropertyId = uuid.uuid1()
g.add((ABROMICS[str(resistanceGenePropertyId)], RDF.type, QB.DimensionProperty))
g.add((ABROMICS[str(resistanceGenePropertyId)], RDFS.label, Literal("Resistance Gene", datatype=XSD.string)))

targetAntibioticPropertyId = uuid.uuid1()
g.add((ABROMICS[str(targetAntibioticPropertyId)], RDF.type, QB.DimensionProperty))
g.add((ABROMICS[str(targetAntibioticPropertyId)], RDFS.label, Literal("Target Antibiotic", datatype=XSD.string)))


# Create slices
targetAntibioticSliceId = uuid.uuid1()
g.add((ABROMICS[str(targetAntibioticSliceId)], RDF.type, QB.Slice))
g.add((ABROMICS[str(targetAntibioticSliceId)], RDFS.label, Literal("slice by antibiotic target", datatype=XSD.string)))


# Create slice key
targetAntibioticSliceKeyId = uuid.uuid1()
g.add((ABROMICS[str(targetAntibioticSliceKeyId)], RDF.type, QB.SliceKey))
g.add((ABROMICS[str(targetAntibioticSliceKeyId)], RDFS.label, Literal("slice by target antibiotic", datatype=XSD.string)))


# Create concrete slice (the real slice entity that will be linked to a dataset)
concreteTargetAntibioticSliceId = uuid.uuid1()
g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], RDF.type, QB.Slice))
g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], QB.sliceStructure, ABROMICS[str(targetAntibioticSliceId)]))

      
# Create the component specification 
componentSpecificationId = uuid.uuid1()
g.add((ABROMICS[str(componentSpecificationId)], QB.component, ABROMICS[str(resistanceGenePropertyId)]))
g.add((ABROMICS[str(componentSpecificationId)], QB.component, ABROMICS[str(targetAntibioticPropertyId)]))
      

# Create the Dataset Structure
datasetStructureId = uuid.uuid1()
g.add((ABROMICS[str(datasetStructureId)], RDF.type, QB.DataStructureDefinition))
g.add((ABROMICS[str(datasetStructureId)], QB.Component, ABROMICS[str(componentSpecificationId)]))


# Create the dataset
datasetId = uuid.uuid1()
g.add((ABROMICS[str(datasetId)], RDF.type, QB.DataSet))
g.add((ABROMICS[str(datasetId)], DCT.title, Literal("Analysis results report", datatype=XSD.string)))
g.add((ABROMICS[str(datasetId)], RDFS.label, Literal("Analysis results report", datatype=XSD.string)))
g.add((ABROMICS[str(datasetId)], RDFS.comment, Literal("Comments about the dataset", datatype=XSD.string)))
g.add((ABROMICS[str(datasetId)], QB.sliceKey, ABROMICS[str(targetAntibioticSliceKeyId)]))
g.add((ABROMICS[str(datasetId)], QB.slice, ABROMICS[str(concreteTargetAntibioticSliceId)]))


# Creation of all the observations of the two rows (Resistance gene and target antibiotic)
for gene in reports[0]["sections"][2]["data"][0]["values"][0]:
    geneObservationId = uuid.uuid1()
    g.add((ABROMICS[str(geneObservationId)], RDF.type, QB.Observation))
    g.add((ABROMICS[str(geneObservationId)], QB.dataSet, ABROMICS[str(datasetId)]))
    g.add((ABROMICS[str(geneObservationId)], RDFS.label, Literal(gene, datatype=XSD.string)))
    # add the observation to the concrete slice
    g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], QB.observation, ABROMICS[str(geneObservationId)]))

for antibiotic in reports[0]["sections"][2]["data"][0]["values"][8]:
    antibioticObservationId = uuid.uuid1()
    g.add((ABROMICS[str(antibioticObservationId)], RDF.type, QB.Observation))
    g.add((ABROMICS[str(antibioticObservationId)], QB.dataSet, ABROMICS[str(datasetId)]))
    g.add((ABROMICS[str(antibioticObservationId)], RDFS.label, Literal(antibiotic, datatype=XSD.string)))
    # add the observation to the concrete slice
    g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], QB.observation, ABROMICS[str(antibioticObservationId)]))

g.serialize(destination="antibiotic-resistance-qb.ttl", format='ttl')

g

#### **Graph of the theorical QB instancitation**

<table><tr>
<td> <img src="img/qb-impl-ex.png" style="width: 1250px; height:1000px"/> </td>
<td> <img src="img/qb-ttl-ex.png" style="width: 500px; height:1000px"/> </td>
</tr></table>

#### **Concrete instanciation (the graph created by the code above)**

The instanciation with concrete data from the ABRomics report is structured as followed when the RDF data cube ontology is used as the graph schema. The name of the entities are harder to understand as every one of them is a unique random string.

<table><tr>
<td> <img src="img/qb-impl-concrete.png" style="width: 1250px; height:1000px"/> </td>
<td> <img src="img/qb-ttl-concrete.png" style="width: 1000px; height:1000px"/> </td>
</tr></table>

## QB Query complexity <a name="QB-queryComplexity"></a>

In [ ]:
# Query that get all the antibiotic resistance gene present in the report and the corresponding molecule the gene is resistant to
query1 = """
    PREFIX ns1: <http://www.abromics.org/ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX dct: <http://purl.org/dc/terms/>
    PREFIX qb: <http://purl.org/linked-data/cube#>
    

    SELECT ?geneNames ?moleculeNames
    WHERE {
        ?dataset a qb:DataSet . 
        ?geneNames a qb:Observation ;
            rdfs:label "Target antibiotic" .
        ?moleculeNames a qb:Observation ; 
            rdfs:label "Resistance gene"
    }
"""

# Query that count the number of entities in the graph
query2 = """
    SELECT ?s ?p ?o 
    WHERE {
        ?s ?p ?o .
    }
"""


queries = [
    {
        "description": "Get all the resistance genes and the resistance molecule",
        "content": query1
    },
    {
        "description": "all the nodes in the graph",
        "content": query2
    }
]

#### **Observations about the query complexity**

Performing a research is strait forward when using filtering the only the observations with a certain label. Using a slice is not necessary here but if the cube was bigger it would have been important to fix the the **resistanceGene** slice and the **targetAntibiotic** slice

## QB Query performance <a name="QB-queryPerformance"></a>

In [ ]:
## Using benchmark class with RDF data cube graph and a query

benchmark = Benchmark(g, queries)

benchmark.launch()
benchmark.displayResults()

## QB Link to other ontologies <a name="QB-linkToOtherOntologies"></a>

The class Observation of RDF data cube comes from **scovo:Item** (SCOVO: Statistical Core Vocabulary). The link to the documentation for this ontology is broken and there is not much more information about the SCOVO ontology online and the link to find the reference graph is broken too. Thus it might be impossible to make a link from an observation to annotations from specific ontologies such as ARO. Linking observations with the vocabulary present in ARO might will require to use a custom property defined locally by in the Abromics-kg project. This is not optimal because it's better to use terms that are already defined in an ontology

## Observations<a name="QB-observations"></a>

The main strength of the QB ontology is to represent the data cubes (data having multiple dimensions). By providing a way to slice the dataset by fixing two dimensions, QB is great for statistical data in a very large dataset. However the ABRomics reports are rather small and two dimentional (a list of creteria for every strain found by the analysis workflow). This means that we can't power the strength of slices execpt if we bundle all the reports into the same very large dataset.

Bundling all the report together in the same dataset will need to consider many other problematics (how to filter public and private data ? what to do if a report is incomplete ?). The major concern of this approach is the semantic complexity. The vocabulary of RDF data cube is complex and it's difficult to have a clear representation of the data in head as a data cube can have n dimensions. In the long run, using RDF data cube might cause more confusion than good. 

Thus, even though it's possible to use RDF data cube to store ABRomics report data and link this data with specific ontology for annotation, we would prefer using an ontology that better data collection workflow of ABRomics rather than staying with highly data oriented ontologies like RDF data cube that will inevitably confuse the other developpers and the average ABRomics user.

#####################################################################################################################

## Create an ABRomics graph using SOSA as the core ontology 

Core ontology : 
- [SOSA / SSN](https://www.w3.org/TR/vocab-ssn/)

Specific ontologies :
- [NCIT](https://bioportal.bioontology.org/ontologies/NCIT)
- [OHMI](https://bioportal.bioontology.org/ontologies/OHMI)
- [ARO](https://bioportal.bioontology.org/ontologies/ARO)
- [GO](https://bioportal.bioontology.org/ontologies/GO)

In [4]:
g = Graph()
    
# Placeholder namespaces
ABROMICS = Namespace("http://www.abromics.org/ontology/")
SOSA = Namespace('http://www.w3.org/ns/sosa/')
NCIt = Namespace('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl/')
OHMI = Namespace('https://bioportal.bioontology.org/ontologies/OHMI/')
ARO = Namespace('https://bioportal.bioontology.org/ontologies/ARO/')
GO = Namespace('https://bioportal.bioontology.org/ontologies/GO/')

## Namespaces used in Uniprot
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")
UP = Namespace("http://purl.uniprot.org/core/")


## Sample metadata
sampleMetadata = reports[0]["sections"][0]["data"][0]["values"]
sampleMetadata

## Storing the sample
sampleId = uuid.uuid1()
g.add((ABROMICS[str(sampleId)], RDF.type, SOSA.ObservableProperty))
g.add((ABROMICS[str(sampleId)], ABROMICS.MicroOrganism, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][1]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.CollectedAt, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][2])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.Source, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][3])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SampleType, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][4])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.Host, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][5]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.Location, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][6])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SequencingTechnology, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][7]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SequencingPlatform, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][8]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SubmitterName, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][9]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SubmitterEmail, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][10]).replace(' ', '_')]))

## Storing the analysis
analysisId = uuid.uuid1()
g.add((ABROMICS[str(analysisId)], RDF.type, SOSA.Observation))
g.add((ABROMICS[str(analysisId)], SOSA.ObservedProperty, ABROMICS[str(sampleId)]))

## Storing the FeatureOfInterest
featureOfInterestId = uuid.uuid1()
g.add((ABROMICS[str(featureOfInterestId)], SOSA.isFeatureOfInterest, ABROMICS[str(analysisId)]))
g.add((ABROMICS[str(featureOfInterestId)], RDF.type, SOSA.FeatureOfInterest))
g.add((ABROMICS[str(featureOfInterestId)], RDF.type, ABROMICS.Genomic))

## Storing the Microbiome
microbiomeId = uuid.uuid1()
g.add((ABROMICS[str(microbiomeId)], RDF.type, OHMI.HumanMicrobiome))
g.add((ABROMICS[str(featureOfInterestId)], ABROMICS.has, ABROMICS[str(microbiomeId)]))

## Store a single strain observed in the sample
strainId = uuid.uuid1()
g.add((ABROMICS[str(strainId)], RDF.type, ARO.Strain))
g.add((ABROMICS[str(microbiomeId)], ABROMICS.contains, ABROMICS[str(strainId)]))
g.add((ABROMICS[str(strainId)], ABROMICS.MLSTProfile, Literal("MLST_profile", datatype=XSD.string)))
g.add((ABROMICS[str(strainId)], ABROMICS.STProfile, Literal("ST_profile", datatype=XSD.string)))

## Store a molecule the strain is resistant to
moleculeId = uuid.uuid1()
g.add((ABROMICS[str(moleculeId)], RDF.type, ARO.Molecule))
g.add((ABROMICS[str(strainId)], ABROMICS.isResistantTo, ABROMICS[str(moleculeId)]))
g.add((ABROMICS[str(moleculeId)], RDFS.label, Literal(str(reports[0]["sections"][2]["data"][0]["values"][8][0]), datatype=XSD.string))) ## store the name of the molecule


## Store the resistance gene associated with the molecule the bacteria is resistant to
geneId = uuid.uuid1()
g.add((ABROMICS[str(geneId)], RDF.type, UP.Gene))
g.add((ABROMICS[str(geneId)], SKOS.prefName, Literal("aadA13", datatype=XSD.string)))
g.add((ABROMICS[str(geneId)], ABROMICS.givesResistanceTo, ABROMICS[str(moleculeId)]))


g.serialize(destination="antibiotic-resistance-sosa.ttl", format='ttl')
    
g

<Graph identifier=Nd541e277ebae417d95586a7d614d83ce (<class 'rdflib.graph.Graph'>)>

In [6]:
## Query 1 :
query1 = """
    PREFIX ns1: <http://www.abromics.org/ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX aro: <https://bioportal.bioontology.org/ontologies/ARO/>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX up: <http://purl.uniprot.org/core/>

    SELECT ?geneName ?moleculeName
    WHERE {
        ?gene a up:Gene .
        ?gene skos:prefName ?geneName .
        ?molecule a aro:Molecule .
        ?gene ns1:givesResistanceTo ?molecule .
        ?molecule rdfs:label ?moleculeName .
    }
"""

query2 = """
    SELECT ?s ?p ?o 
    WHERE {
        ?s ?p ?o .
    }
"""


queries = [
    {
        "description": "Get all the resistance genes and the resistance molecule",
        "content": query1
    },
    {
        "description": "all the nodes in the graph",
        "content": query2
    }
]

In [6]:
## Using benchmark class with SOSA graph and a query

benchmark = Benchmark(g, queries)

benchmark.launch()
benchmark.displayResults()

                                                                 query 1  \
description            Get all the resistance genes and the resistanc...   
avg_elapsed_time (ms)                                               4.21   
max_elapsed_time (ms)                                              69.49   
min_elapse_time (ms)                                                3.15   

                                          query 2  
description            all the nodes in the graph  
avg_elapsed_time (ms)                        1.54  
max_elapsed_time (ms)                        1.75  
min_elapse_time (ms)                         1.49  


## Create an ABRomics graph using RDF Datacube as the core ontology

Core ontology:
- [RDF Data Cube](https://www.w3.org/TR/vocab-data-cube/)

Insipired by the graph example presented in section 6 of the RDF data cube documentation



In [8]:
## Query 1 :
query1 = """
    PREFIX ns1: <http://www.abromics.org/ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX dct: <http://purl.org/dc/terms/>
    PREFIX qb: <http://purl.org/linked-data/cube#>
    

    SELECT ?geneNames ?moleculeNames
    WHERE {
        ?dataset a qb:DataSet . 
        ?geneNames a qb:Observation ;
            rdfs:label "Target antibiotic" .
        ?moleculeNames a qb:Observation ; 
            rdfs:label "Resistance gene"
    }
"""

query2 = """
    SELECT ?s ?p ?o 
    WHERE {
        ?s ?p ?o .
    }
"""


queries = [
    {
        "description": "Get all the resistance genes and the resistance molecule",
        "content": query1
    },
    {
        "description": "all the nodes in the graph",
        "content": query2
    }
]

In [7]:
## Using benchmark class with RDF data cube graph and a query

benchmark = Benchmark(g, queries)

benchmark.launch()
benchmark.displayResults()

                                                                 query 1  \
description            Get all the resistance genes and the resistanc...   
avg_elapsed_time (ms)                                               4.41   
max_elapsed_time (ms)                                              68.67   
min_elapse_time (ms)                                                3.24   

                                          query 2  
description            all the nodes in the graph  
avg_elapsed_time (ms)                        1.63  
max_elapsed_time (ms)                         1.9  
min_elapse_time (ms)                         1.54  


## Differences between SOSA and RDF data cube

From a performance perspective, querying a graph using SOSA or RDF data cube is similar, slighly faster for RDF data cube (1ms faster) when not using any slices in the dataset.
However SOSA is a bit easier to query as the entities to manipulate such as Observation, Sample and ObservableProperties are much more concrete than the sometimes confusing ComponentProperties, Slices, SliceKeys
of RDF data cube.

With RDF data cube the structure of the data needs to be well understood by the user before performing a query, even more if the dataset have to be sliced during the query. 

## Conclusion

In term of performance, it's yet not possible to know which ontology is more suitable.
Further investigation about dataset slice have to be done for RDF data cube. More complex queries need to be performed on both graphs to compare performance objectively. 

# Comparison of SOSA and RDF data cube schema

SOSA and RDF data cube are two generic ontologies are opinionated in the way the data should be stored. Then, the goal of this section is to compare the ease of storing typical ABRomics data (ABRomics reports) in both a SOSA graph and an graph created by using RDF data cube as the generic ontology

- Storing data using QB
    - Storing an ABRomics report
    - Performances
- Storing data using SOSA/SSN
    - Storing an ABRomics report
    - Storing informations about the workflow that generated a report
    - Annotation of report with ARO
    - Performances
- Conclusion

In [1]:
## Necessary imports

import json
import time
import pandas as pd

from rdflib import Graph
import uuid
from rdflib import Namespace
from rdflib import FOAF, DCTERMS, SKOS, VANN, TIME, RDF, RDFS, XSD, OWL, Literal

## Storing data using RDF data cube (QB)

The QB ontology is used to store large dataset that could be represented as a matrix of n dimensions. Oriented towards statistical usecases, RDF data cube define different classes to interact with the data cube using slice. Filters can also be applied to the properties that are linked to the slice or to the dataset structure definition

![QB_schema](img/qb-schema.png)

## Storing an ABRomics report

The ABRomics report can be stored in a RDF data cube data set where the content of the data is stored in many observations.
As the ABRomics report is a simple table using, defining slices is useless because slices are used to fix two dimensions of the data cube to extract a table that can later be analysed.

The code below perform an instanciation of QB schema by storing the resistance genes and target antibiotic found in the ABRomics report. The same storing strategy can be applied to any data present in the ABRomics report.  

In [7]:
g = Graph()

ABROMICS = Namespace("http://www.abromics.org/ontology/")
RDFS = Namespace("http://www.w3.org/2000/01/rdf-schema#")
DCT = Namespace("http://purl.org/dc/terms/")

QB = Namespace("http://purl.org/linked-data/cube#")

# Read part 6 of the RDF data cube documentation to understand how such data should be encoded

analysisResult = pd.DataFrame(reports[0]["sections"][2]["data"][0]["values"], index=reports[0]["sections"][2]["data"][0]["header"])

# Create all the dimensions and measures
resistanceGenePropertyId = uuid.uuid1()
g.add((ABROMICS[str(resistanceGenePropertyId)], RDF.type, QB.DimensionProperty))
g.add((ABROMICS[str(resistanceGenePropertyId)], RDFS.label, Literal("Resistance Gene", datatype=XSD.string)))

targetAntibioticPropertyId = uuid.uuid1()
g.add((ABROMICS[str(targetAntibioticPropertyId)], RDF.type, QB.DimensionProperty))
g.add((ABROMICS[str(targetAntibioticPropertyId)], RDFS.label, Literal("Target Antibiotic", datatype=XSD.string)))


# Create slices
targetAntibioticSliceId = uuid.uuid1()
g.add((ABROMICS[str(targetAntibioticSliceId)], RDF.type, QB.Slice))
g.add((ABROMICS[str(targetAntibioticSliceId)], RDFS.label, Literal("slice by antibiotic target", datatype=XSD.string)))


# Create slice key
targetAntibioticSliceKeyId = uuid.uuid1()
g.add((ABROMICS[str(targetAntibioticSliceKeyId)], RDF.type, QB.SliceKey))
g.add((ABROMICS[str(targetAntibioticSliceKeyId)], RDFS.label, Literal("slice by target antibiotic", datatype=XSD.string)))


# Create concrete slice (the real slice entity that will be linked to a dataset)
concreteTargetAntibioticSliceId = uuid.uuid1()
g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], RDF.type, QB.Slice))
g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], QB.sliceStructure, ABROMICS[str(targetAntibioticSliceId)]))

      
# Create the component specification 
componentSpecificationId = uuid.uuid1()
g.add((ABROMICS[str(componentSpecificationId)], QB.component, ABROMICS[str(resistanceGenePropertyId)]))
g.add((ABROMICS[str(componentSpecificationId)], QB.component, ABROMICS[str(targetAntibioticPropertyId)]))
      

# Create the Dataset Structure
datasetStructureId = uuid.uuid1()
g.add((ABROMICS[str(datasetStructureId)], RDF.type, QB.DataStructureDefinition))
g.add((ABROMICS[str(datasetStructureId)], QB.Component, ABROMICS[str(componentSpecificationId)]))


# Create the dataset
datasetId = uuid.uuid1()
g.add((ABROMICS[str(datasetId)], RDF.type, QB.DataSet))
g.add((ABROMICS[str(datasetId)], DCT.title, Literal("Analysis results report", datatype=XSD.string)))
g.add((ABROMICS[str(datasetId)], RDFS.label, Literal("Analysis results report", datatype=XSD.string)))
g.add((ABROMICS[str(datasetId)], RDFS.comment, Literal("Comments about the dataset", datatype=XSD.string)))
g.add((ABROMICS[str(datasetId)], QB.sliceKey, ABROMICS[str(targetAntibioticSliceKeyId)]))
g.add((ABROMICS[str(datasetId)], QB.slice, ABROMICS[str(concreteTargetAntibioticSliceId)]))


# Creation of all the observations of the two rows (Resistance gene and target antibiotic)
for gene in reports[0]["sections"][2]["data"][0]["values"][0]:
    geneObservationId = uuid.uuid1()
    g.add((ABROMICS[str(geneObservationId)], RDF.type, QB.Observation))
    g.add((ABROMICS[str(geneObservationId)], QB.dataSet, ABROMICS[str(datasetId)]))
    g.add((ABROMICS[str(geneObservationId)], RDFS.label, Literal(gene, datatype=XSD.string)))
    # add the observation to the concrete slice
    g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], QB.observation, ABROMICS[str(geneObservationId)]))

for antibiotic in reports[0]["sections"][2]["data"][0]["values"][8]:
    antibioticObservationId = uuid.uuid1()
    g.add((ABROMICS[str(antibioticObservationId)], RDF.type, QB.Observation))
    g.add((ABROMICS[str(antibioticObservationId)], QB.dataSet, ABROMICS[str(datasetId)]))
    g.add((ABROMICS[str(antibioticObservationId)], RDFS.label, Literal(antibiotic, datatype=XSD.string)))
    # add the observation to the concrete slice
    g.add((ABROMICS[str(concreteTargetAntibioticSliceId)], QB.observation, ABROMICS[str(antibioticObservationId)]))

g.serialize(destination="antibiotic-resistance-qb.ttl", format='ttl')

g

# Checkout the handmade draft on the desk to make a decision on the structure of the KG using RDF data cube

<Graph identifier=Ncadcaad7baed41d3baf0c1b4f6cd4739 (<class 'rdflib.graph.Graph'>)>

## Turtle file result and instanciation

The code above creates a .ttl file containing an instanciation of QB using ABRomics data.

### Instanciation example

Theorical turtle file to store data from the ABRomics report into a graph with QB as the ontology used to organise the data

<table><tr>
<td> <img src="img/qb-impl-ex.png" style="width: 1250px; height:1000px"/> </td>
<td> <img src="img/qb-ttl-ex.png" style="width: 500px; height:1000px"/> </td>
</tr></table>

### Concrete instanciation (the graph created by the code above)

The instanciation with concrete data from the ABRomics report is structured as followed when the RDF data cube ontology is used as the graph schema. The name of the entities are harder to understand as every one of them is a unique random string.

<table><tr>
<td> <img src="img/qb-impl-concrete.png" style="width: 1250px; height:1000px"/> </td>
<td> <img src="img/qb-ttl-concrete.png" style="width: 1000px; height:1000px"/> </td>
</tr></table>

## Observations

The main strength of the QB ontology is to represent the data cubes (data having multiple dimensions). By providing a way to slice the dataset by fixing two dimensions, QB is great for statistical data in a very large dataset. However the ABRomics reports are rather small and two dimentional (a list of creteria for every strain found by the analysis workflow). This means that we can't power the strength of slices execpt if we bundle all the reports into the same very large dataset.

Bundling all the report together in the same dataset will need to consider many other problematics (how to filter public and private data ? what to do if a report is incomplete ?). The major concern of this approach is the semantic complexity. The vocabulary of RDF data cube is complex and it's difficult to have a clear representation of the data in head as a data cube can have n dimensions. In the long run, using RDF data cube might cause more confusion than good. 

Thus, even though it's possible to use RDF data cube to store ABRomics report data and link this data with specific ontology for annotation, we would prefer using an ontology that better data collection workflow of ABRomics rather than staying with highly data oriented ontologies like RDF data cube that will inevitably confuse the other developpers and the average ABRomics user.

## Storing data using a SOSA/SSN

The SOSA ontology define a standard vocabulary allowing captors to communicate with the same semantic terms within a network. The ontology define what is an Observation a FeatureOfInterest and how measues can be carried out by a sample. There is also 3 different schemas in SOSA/SSN **(Observation, Actuation, Sampling)** that we can choose from to better represent the ABRomics data. 